# Machine Learning

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import log_loss, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import xgboost as xgb
from scipy.stats import (
    uniform, 
    randint,
    loguniform
)
import sys
import timm
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, random_split, DataLoader, Subset, ConcatDataset
import copy
from pprint import pprint
import json
import os
import warnings
import lightgbm as lgb
import optuna

## Baseline Model (XGBoost)

In [ ]:
initial_df = pd.read_parquet("../data/processed/anime_data_2.parquet")
initial_df.info()

In [ ]:
df = initial_df.head(5278)
real_df = initial_df.tail(72)

print(df.info())
print(real_df.info())

## Input Preparation

Right now, the priority is to reduce dimensions. The plan is the following:
* Reduce the dimensions of the image tensors from 512
* Reduce the dimensions of sentimental analysis tensors from 768
* Reduce the pool of producers and studios into embedded vectors

In [ ]:
df = df.reset_index(drop=True)

all_indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=42
)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.10,
    random_state=42
)

print(df.index[:5])
print(train_idx[:5])

### Some Continuous Variables

In [ ]:
df.info()

In [ ]:
continuous_features = ['prequel_score', 'prequel_members']

for feature in continuous_features:
    cts_scaler = StandardScaler()
    real_cts_scaler = StandardScaler()

    cts_scaler.fit(
        df.loc[train_idx, [feature]]
    )

    df[feature] = cts_scaler.transform(
        df[[feature]]
    )

    real_df[feature] = real_cts_scaler.fit_transform(
        real_df[[feature]]
    )

print(df['prequel_score'].describe())
print(df['prequel_members'].describe())
real_df['prequel_members'].describe()

### Studios & Producers

In [ ]:
studio_to_idx = {"<UNK>": 0}
producer_to_idx = {"<UNK>": 0}

for studios in df.iloc[train_idx]["studios"]:
    for studio in studios:
        if studio not in studio_to_idx:
            studio_to_idx[studio] = len(studio_to_idx)

for producers in df.iloc[train_idx]["producers"]:
    for producer in producers:
        if producer not in producer_to_idx:
            producer_to_idx[producer] = len(producer_to_idx)

n_studios = len(studio_to_idx)
n_producers = len(producer_to_idx)

print("Number of studios:", n_studios)
print("Number of producers:", n_producers)

In [ ]:
def get_studio_indices(studios):
    return [
        studio_to_idx.get(studio, 0)
        for studio in studios
    ]

def get_producer_indices(producers):
    return [
        producer_to_idx.get(producer, 0)
        for producer in producers
    ]

df["studio_idx"] = df["studios"].apply(get_studio_indices)
df["producer_idx"] = df["producers"].apply(get_producer_indices)

real_df["studio_idx"] = real_df["studios"].apply(get_studio_indices)
real_df["producer_idx"] = real_df["producers"].apply(get_producer_indices)

print(df["studio_idx"].head())
print(df["producer_idx"].head())

In [ ]:
def create_embedding_bag_inputs(index_lists):
    flat_indices = []
    offsets = []

    current_offset = 0

    for indices in index_lists:
        offsets.append(current_offset)
        flat_indices.extend(indices)
        current_offset += len(indices)

    return (
        torch.tensor(flat_indices, dtype=torch.long),
        torch.tensor(offsets, dtype=torch.long)
    )


studio_indices, studio_offsets = create_embedding_bag_inputs(
    df["studio_idx"]
)

producer_indices, producer_offsets = create_embedding_bag_inputs(
    df["producer_idx"]
)

real_studio_indices, real_studio_offsets = create_embedding_bag_inputs(
    real_df["studio_idx"]
)

real_producer_indices, real_producer_offsets = create_embedding_bag_inputs(
    real_df["producer_idx"]
)

print("Studio indices:", studio_indices.shape)
print("Studio offsets:", studio_offsets.shape)

print("Producer indices:", producer_indices.shape)
print("Producer offsets:", producer_offsets.shape)

In [ ]:
def split_embedding_bag_inputs(index_lists, train_idx, val_idx, test_idx):
    
    def create_for_rows(rows):
        selected_lists = [index_lists[i] for i in rows]
        return create_embedding_bag_inputs(selected_lists)

    train_indices, train_offsets = create_for_rows(train_idx)
    val_indices, val_offsets = create_for_rows(val_idx)
    test_indices, test_offsets = create_for_rows(test_idx)

    return (
        train_indices, train_offsets,
        val_indices, val_offsets,
        test_indices, test_offsets
    )


(
    studio_indices_train,
    studio_offsets_train,
    studio_indices_val,
    studio_offsets_val,
    studio_indices_test,
    studio_offsets_test
) = split_embedding_bag_inputs(
    df["studio_idx"].tolist(),
    train_idx,
    val_idx,
    test_idx
)


(
    producer_indices_train,
    producer_offsets_train,
    producer_indices_val,
    producer_offsets_val,
    producer_indices_test,
    producer_offsets_test
) = split_embedding_bag_inputs(
    df["producer_idx"].tolist(),
    train_idx,
    val_idx,
    test_idx
)

### Synopsis

In [ ]:
initial_semantic_embeddings = np.load('../data/processed/semantic_embeddings.npy')
print(initial_semantic_embeddings.shape)
semantic_embeddings = initial_semantic_embeddings[:5278]
real_semantic_embeddings = initial_semantic_embeddings[-72:]

In [ ]:
text_scaler = StandardScaler()

text_scaler.fit(
    semantic_embeddings[train_idx]
)

semantic_embeddings_train = text_scaler.transform(
    semantic_embeddings[train_idx]
)

semantic_embeddings_val = text_scaler.transform(
    semantic_embeddings[val_idx]
)

semantic_embeddings_test = text_scaler.transform(
    semantic_embeddings[test_idx]
)

real_semantic_embeddings = text_scaler.transform(
    real_semantic_embeddings
)

print(
    "Train text NaNs:",
    np.isnan(semantic_embeddings_train).sum()
)

print(
    "Val text NaNs:",
    np.isnan(semantic_embeddings_val).sum()
)

print(
    "Test text NaNs:",
    np.isnan(semantic_embeddings_test).sum()
)

print(
    "Train text infs:",
    np.isinf(semantic_embeddings_train).sum()
)

In [ ]:
class LearnedProjector(nn.Module):
    """
    Projects a frozen all-mpnet-base-v2 sentence embedding (768-dim)
    down to a smaller learned representation via a linear layer.

    Note: sentence-transformers' mpnet output is L2-normalized by default
    (normalize_embeddings=True), so no extra normalization is applied here
    on the input side.

    Usage:
        projector = LearnedProjector(in_dim=768, out_dim=64)
        z = projector(x)  # x: (batch, 768) -> z: (batch, 64)
    """
    def __init__(self, in_dim: int = 768, out_dim: int = 64,
                 hidden_dim: int | None = None, dropout: float = 0.1):
        super().__init__()

        if hidden_dim is None:
            self.net = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.LayerNorm(out_dim)
            )
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim),
                nn.LayerNorm(out_dim)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

projector = LearnedProjector(in_dim=768, out_dim=64)

## Images

In [ ]:
image_data = np.load("../data/processed/image_embeddings.npy")

In [ ]:
image_scaler = StandardScaler()

image_scaler.fit(
    image_data[train_idx]
)

image_train = image_scaler.transform(
    image_data[train_idx]
)

image_val = image_scaler.transform(
    image_data[val_idx]
)

image_test = image_scaler.transform(
    image_data[test_idx]
)

image_real = image_scaler.transform(
    image_data[-72:]
)

print("Image train NaNs:", np.isnan(image_train).sum())
print("Image val NaNs:", np.isnan(image_val).sum())
print("Image test NaNs:", np.isnan(image_test).sum())

In [ ]:
print(df.info())
print(real_df.info())

In [ ]:
X_other_pre = df.drop(columns=['members', 'title', 'studio_idx', 'producer_idx', 'mal_id', 'cohort', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'dropped_z', 'forum_z'])
X_other_pre = X_other_pre.reset_index(drop=True)
X_other_pre = pd.get_dummies(X_other_pre, columns=['rating', 'prequel_type'], dtype=int)
X_other_pre.columns = X_other_pre.columns.str.replace(' ', '_')
print(X_other_pre.info())

real_other_pre = real_df.drop(columns=['members', 'title', 'studio_idx', 'producer_idx', 'mal_id', 'cohort', 'producers', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'dropped_z', 'forum_z'])
real_other_pre = real_other_pre.reset_index(drop=True)
real_other_pre = pd.get_dummies(real_other_pre, columns=['rating', 'prequel_type'], dtype=int)
real_other_pre[[f'prequel_type_{type}' for type in ['Movie', 'Music', 'ONA', 'OVA', 'PV', 'Special', 'TV_Special']]] = 0
real_other_pre['rating_R+_-_Mild_Nudity'] = 0
real_other_pre.columns = real_other_pre.columns.str.replace(' ', '_')
print(real_other_pre.info())

In [ ]:
other_scaler = StandardScaler()

features = ['adaptation_score', 'adaptation_members']

print(X_other_pre[features].isna().sum())

train_data = X_other_pre.iloc[train_idx][features]
val_data   = X_other_pre.iloc[val_idx][features]
test_data  = X_other_pre.iloc[test_idx][features]
real_data = real_other_pre[features]

imputer = SimpleImputer(strategy='mean') # or 'median'
train_imputed = imputer.fit_transform(train_data)
val_imputed   = imputer.transform(val_data)
test_imputed  = imputer.transform(test_data)
real_imputed = imputer.transform(real_data)

adaptation_train = other_scaler.fit_transform(train_imputed)
adaptation_val   = other_scaler.transform(val_imputed)
adaptation_test  = other_scaler.transform(test_imputed)
adaptation_real = other_scaler.transform(real_imputed)

print(adaptation_train)

In [ ]:
X_other_pre.loc[train_idx, 'adaptation_score'] = adaptation_train[:, 0]
X_other_pre.loc[val_idx, 'adaptation_score'] = adaptation_val[:, 0]
X_other_pre.loc[test_idx, 'adaptation_score'] = adaptation_test[:, 0]
real_other_pre['adaptation_score'] = adaptation_real[:, 0]

X_other_pre.loc[train_idx, 'adaptation_members'] = adaptation_train[:, 1]
X_other_pre.loc[val_idx, 'adaptation_members'] = adaptation_val[:, 1]
X_other_pre.loc[test_idx, 'adaptation_members'] = adaptation_test[:, 1]
real_other_pre['adaptation_members'] = adaptation_real[:, 1]

print(X_other_pre.info())
print(real_other_pre.info())

## Baseline Model (XGBoost)

For the baseline model, we will not be taking studios and producers into account, as it's hard to make an analogous EmbeddingBag for them.

In [ ]:
X_train = pd.concat([X_other_pre.loc[train_idx], pd.DataFrame(image_train, index=train_idx)], axis=1)
X_train['thumbnail'].astype(int)
X_val = pd.concat([X_other_pre.loc[val_idx], pd.DataFrame(image_val, index=val_idx)], axis=1)
X_val['thumbnail'].astype(int)
X_test = pd.concat([X_other_pre.loc[test_idx], pd.DataFrame(image_test, index=test_idx)], axis=1)
X_val['thumbnail'].astype(int)

print(X_train.isna().any().any())
print(X_val.isna().any().any())
print(X_test.isna().any().any())

In [ ]:
y = df['dropped_z']

y_train = y[train_idx]
y_val = y[val_idx]
y_test = y[test_idx]

In [ ]:
def gaussian_nll_objective(preds, train_data):
    y = train_data.get_label()
    n_samples = len(y)
    preds = preds.reshape(n_samples, 2)
    mu = preds[:, 0]
    s = preds[:, 1]  # s = log(sigma^2)
    var = np.exp(s)
    
    res = mu - y
    grad_mu = res / var
    grad_s = 0.5 * (1.0 - (res**2) / var)
    
    hess_mu = 1.0 / var
    hess_s = 0.5 * (res**2) / var
    hess_s = np.maximum(hess_s, 1e-4) # Stabilize Hessian
    
    grad = np.vstack((grad_mu, grad_s)).T.flatten()
    hess = np.vstack((hess_mu, hess_s)).T.flatten()
    return grad, hess

def gaussian_nll_metric(preds, train_data):
    y = train_data.get_label()
    n_samples = len(y)
    preds = preds.reshape(n_samples, 2)
    mu = preds[:, 0]
    s = preds[:, 1]
    
    nll = 0.5 * np.exp(-s) * (y - mu)**2 + 0.5 * s + 0.5 * np.log(2 * np.pi)
    return 'gaussian_nll', np.mean(nll), False

In [ ]:
def objective(trial):
    # Base dictionary setup
    params = {
        'num_class': 2,
        'verbosity': -1,
        'objective': gaussian_nll_objective,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'num_leaves': trial.suggest_int('num_leaves', 7, 23),
        'min_child_samples': trial.suggest_int('min_child_samples', 30, 150),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 0.4),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 50.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 50.0, log=True),
    }

    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X_train):
        if hasattr(X_train, "iloc"):
            X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
            X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
        else:
            X_tr, y_tr = X_train[train_idx], y_train[train_idx]
            X_va, y_va = X_train[val_idx], y_train[val_idx]
        
        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
        
        # Train booster utilizing the modern LightGBM 4.x signature
        bst = lgb.train(
            params,
            train_set=dtrain,
            num_boost_round=400,
            valid_sets=[dval],
            feval=gaussian_nll_metric,  # FIX: feval is still accepted or can use eval_metric
            callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
        )
        
        preds = bst.predict(X_va)
        _, score, _ = gaussian_nll_metric(preds, dval)
        cv_scores.append(score)
        
    return np.mean(cv_scores)


In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)  # Increase n_trials for real production use

print("\n Optimization Finished!")
print(f"Best Gaussian NLL Value: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

print("\nTraining final model on full training set...")
best_params = study.best_params
best_params['num_class'] = 2
best_params['verbosity'] = -1
best_params['objective'] = gaussian_nll_objective

full_train_dataset = lgb.Dataset(X_train, label=y_train)
final_bst = lgb.train(
    best_params,
    train_set=full_train_dataset,
    num_boost_round=500
)

## Fusion Network

In [ ]:
class FusionNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        # # Text projector
        # self.text_projector = LearnedProjector(
        #     in_dim=768,
        #     hidden_dim=128,
        #     out_dim=64,
        #     dropout=0.4
        # )

        # Image branch
        self.image_branch = nn.Sequential(
            nn.Linear(418, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        # Tabular branch
        self.other_branch = nn.Sequential(
            nn.Linear(83, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        self.studio_embedding = nn.EmbeddingBag(
            num_embeddings=n_studios,
            embedding_dim=8,
            mode='mean'
        )

        self.producer_embedding = nn.EmbeddingBag(
            num_embeddings=n_producers,
            embedding_dim=8,
            mode='mean'
        )

        # Fusion
        self.fusion = nn.Sequential(
            nn.Linear(64 + 32 + 8 + 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, image, other, studio_indices, studio_offsets, producer_indices, producer_offsets):
        # 1. Check raw inputs
        for name, tensor in [('image', image), ('other', other)]:
            if torch.isnan(tensor).any():
                print(f"NaN detected in raw input: {name}")

        image_features = self.image_branch(image)
        other_features = self.other_branch(other)
        
        # 2. Check branches
        if torch.isnan(image_features).any(): print("NaN in image branch (Check BatchNorm/Batch Size)")
        if torch.isnan(other_features).any(): print("NaN in other branch")

        studio_features = self.studio_embedding(studio_indices, studio_offsets)
        producer_features = self.producer_embedding(producer_indices, producer_offsets)
        
        # 3. Check embeddings
        if torch.isnan(studio_features).any(): print("NaN in studio embeddings")

        combined = torch.cat([image_features, other_features, studio_features, producer_features], dim=1)
        
        out = self.fusion(combined)
        if torch.isnan(out).any(): print("NaN generated inside Fusion layers")

        mean = out[:, 0]      # shape [32], not [32, 0:1]
        raw_std = out[:, 1]   # shape [32]

        std = torch.exp(raw_std) + 1e-6
        norm_dist = torch.distributions.Normal(mean, std)
        
        return norm_dist

In [ ]:
# X_text_train = torch.from_numpy(
#     semantic_embeddings_train.astype(np.float32)
# )

# X_text_val = torch.from_numpy(
#     semantic_embeddings_val.astype(np.float32)
# )

# X_text_test = torch.from_numpy(
#     semantic_embeddings_test.astype(np.float32)
# )

X_image_train = torch.from_numpy(
    image_train.astype(np.float32)
)

X_image_val = torch.from_numpy(
    image_val.astype(np.float32)
)

X_image_test = torch.from_numpy(
    image_test.astype(np.float32)
)

X_other = torch.from_numpy(
    X_other_pre.to_numpy(dtype=np.float32)
)

y_score = torch.tensor(
    df["dropped_z"].to_numpy(dtype=np.float32) # change depending on metric
)

# real_text = torch.from_numpy(
#     real_semantic_embeddings.astype(np.float32)
# )

real_image = torch.from_numpy(
    image_real.astype(np.float32)
)

real_other = torch.tensor(
    real_other_pre.to_numpy(dtype=np.float32)
)

In [ ]:
other_train = X_other[train_idx]
other_val = X_other[val_idx]
other_test = X_other[test_idx]

score_train = y_score[train_idx]
score_val = y_score[val_idx]
score_test = y_score[test_idx]

In [ ]:
model = FusionNetwork()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


class AnimeDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        image,
        other,
        target,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    ):
        self.image = image
        self.other = other
        self.target = target

        self.studio_indices = studio_indices
        self.studio_offsets = studio_offsets

        self.producer_indices = producer_indices
        self.producer_offsets = producer_offsets

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):

        # Figure out where this anime's studio list starts
        studio_start = self.studio_offsets[idx]

        if idx + 1 < len(self.studio_offsets):
            studio_end = self.studio_offsets[idx + 1]
        else:
            studio_end = len(self.studio_indices)

        studio_indices = self.studio_indices[
            studio_start:studio_end
        ]

        # Same thing for producers
        producer_start = self.producer_offsets[idx]

        if idx + 1 < len(self.producer_offsets):
            producer_end = self.producer_offsets[idx + 1]
        else:
            producer_end = len(self.producer_indices)

        producer_indices = self.producer_indices[
            producer_start:producer_end
        ]

        return (
            self.image[idx],
            self.other[idx],
            studio_indices,
            producer_indices,
            self.target[idx]
        )

def collate_fn(batch):

    images = torch.stack([item[0] for item in batch])
    others = torch.stack([item[1] for item in batch])
    targets = torch.stack([item[4] for item in batch])

    studio_indices = []
    studio_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[2]

        studio_offsets.append(current_offset)

        studio_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    studio_indices = torch.tensor(
        studio_indices,
        dtype=torch.long
    )

    studio_offsets = torch.tensor(
        studio_offsets,
        dtype=torch.long
    )

    producer_indices = []
    producer_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[3]

        producer_offsets.append(current_offset)

        producer_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    producer_indices = torch.tensor(
        producer_indices,
        dtype=torch.long
    )

    producer_offsets = torch.tensor(
        producer_offsets,
        dtype=torch.long
    )

    return (
        images,
        others,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        targets
    )

In [ ]:
train_dataset = AnimeDataset(
    X_image_train,
    other_train,
    score_train,
    studio_indices_train,
    studio_offsets_train,
    producer_indices_train,
    producer_offsets_train
)

val_dataset = AnimeDataset(
    X_image_val,
    other_val,
    score_val,
    studio_indices_val,
    studio_offsets_val,
    producer_indices_val,
    producer_offsets_val
)

test_dataset = AnimeDataset(
    X_image_test,
    other_test,
    score_test,
    studio_indices_test,
    studio_offsets_test,
    producer_indices_test,
    producer_offsets_test
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


In [ ]:
def nll_loss(dist, target):
    # print(dist.log_prob(target).shape)
    return -dist.log_prob(target).mean()

In [ ]:
full_dataset = ConcatDataset([train_dataset, val_dataset])
n_samples = len(full_dataset)

k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

batch_size = train_loader.batch_size
collate_fn = train_loader.collate_fn 
patience = 15
n_epochs = 100

fold_results = []          
fold_model_states = []    

for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(n_samples))):

    print(f"\n===== Fold {fold + 1}/{k} =====")

    model = FusionNetwork().to(device)  
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,
        weight_decay=1e-2
    )

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )
    fold_val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    for epoch in range(n_epochs):

        model.train()
        train_loss = 0

        for (
            image, other,
            studio_indices, studio_offsets,
            producer_indices, producer_offsets,
            target
        ) in fold_train_loader:

            image = image.to(device)
            other = other.to(device)
            studio_indices = studio_indices.to(device)
            studio_offsets = studio_offsets.to(device)
            producer_indices = producer_indices.to(device)
            producer_offsets = producer_offsets.to(device)
            target = target.to(device)

            prediction = model(
                image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets
            )

            loss = nll_loss(prediction, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(fold_train_loader)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for (
                image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets,
                target
            ) in fold_val_loader:

                image = image.to(device)
                other = other.to(device)
                studio_indices = studio_indices.to(device)
                studio_offsets = studio_offsets.to(device)
                producer_indices = producer_indices.to(device)
                producer_offsets = producer_offsets.to(device)
                target = target.to(device)

                prediction = model(
                    image, other,
                    studio_indices, studio_offsets,
                    producer_indices, producer_offsets
                )

                loss = nll_loss(prediction, target)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(fold_val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Fold {fold + 1} early stopping at epoch {epoch}.")
                break

        if epoch % 5 == 0 or patience_counter == 0:
            print(
                f"  Epoch {epoch}: "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

    print(f"Fold {fold + 1} best val loss: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)
    fold_model_states.append(best_model_state)

fold_results = np.array(fold_results)
print(f"\n===== CV Results ({k}-fold) =====")
print(f"Per-fold val loss: {fold_results}")
print(f"Mean: {fold_results.mean():.4f}  |  Std: {fold_results.std():.4f}")

best_fold_idx = fold_results.argmin()
best_model_state = fold_model_states[best_fold_idx]
model = FusionNetwork().to(device)
model.load_state_dict(best_model_state)
print(f"\nLoaded weights from fold {best_fold_idx + 1} (val loss {fold_results[best_fold_idx]:.4f})")

In [ ]:
model.eval()

total_nll = 0
total_samples = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for (
        image,
        other,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        target
    ) in test_loader:

        image = image.to(device)
        other = other.to(device)

        studio_indices = studio_indices.to(device)
        studio_offsets = studio_offsets.to(device)

        producer_indices = producer_indices.to(device)
        producer_offsets = producer_offsets.to(device)

        target = target.to(device)

        predictions = model(
            image,
            other,
            studio_indices,
            studio_offsets,
            producer_indices,
            producer_offsets
        )

        nll = nll_loss(predictions, target)

        total_nll += nll * 32
        total_samples += target.size(0)



final_nll = (
    total_nll /
    total_samples
)


print(f"Test NLL:  {final_nll:.4f}")

### Current Anime Prediction

In [ ]:
# data prep

In [ ]:
class AnimeInferenceDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        image,
        other,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    ):
        self.image = image
        self.other = other

        self.studio_indices = studio_indices
        self.studio_offsets = studio_offsets

        self.producer_indices = producer_indices
        self.producer_offsets = producer_offsets

    def __len__(self):
        return len(self.image)

    def __getitem__(self, idx):

        studio_start = self.studio_offsets[idx]

        if idx + 1 < len(self.studio_offsets):
            studio_end = self.studio_offsets[idx + 1]
        else:
            studio_end = len(self.studio_indices)

        studio_indices = self.studio_indices[
            studio_start:studio_end
        ]

        producer_start = self.producer_offsets[idx]

        if idx + 1 < len(self.producer_offsets):
            producer_end = self.producer_offsets[idx + 1]
        else:
            producer_end = len(self.producer_indices)

        producer_indices = self.producer_indices[
            producer_start:producer_end
        ]

        return (
            self.image[idx],
            self.other[idx],
            studio_indices,
            producer_indices
        )

def inference_collate_fn(batch):


    images = torch.stack(
        [item[0] for item in batch]
    )

    others = torch.stack(
        [item[1] for item in batch]
    )

    # -------------------------
    # Studios
    # -------------------------

    studio_indices = []
    studio_offsets = []

    current_offset = 0

    for item in batch:

        indices = item[2]

        studio_offsets.append(current_offset)

        studio_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    studio_indices = torch.tensor(
        studio_indices,
        dtype=torch.long
    )

    studio_offsets = torch.tensor(
        studio_offsets,
        dtype=torch.long
    )

    # -------------------------
    # Producers
    # -------------------------

    producer_indices = []
    producer_offsets = []

    current_offset = 0

    for item in batch:

        indices = item[3]

        producer_offsets.append(current_offset)

        producer_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    producer_indices = torch.tensor(
        producer_indices,
        dtype=torch.long
    )

    producer_offsets = torch.tensor(
        producer_offsets,
        dtype=torch.long
    )

    return (
        images,
        others,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    )

In [ ]:
real_dataset = AnimeInferenceDataset(
    real_image,
    real_other,
    real_studio_indices,
    real_studio_offsets,
    real_producer_indices,
    real_producer_offsets
)

In [ ]:
real_loader = DataLoader(
    real_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=inference_collate_fn
)

In [ ]:
model.eval()

means = []
stds = []

with torch.no_grad():

    for (
        image,
        other,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    ) in real_loader:

        image = image.to(device)
        other = other.to(device)

        studio_indices = studio_indices.to(device)
        studio_offsets = studio_offsets.to(device)

        producer_indices = producer_indices.to(device)
        producer_offsets = producer_offsets.to(device)

        output = model(
            image,
            other,
            studio_indices,
            studio_offsets,
            producer_indices,
            producer_offsets
        )

        means.append(
            output.mean
        )

        stds.append(
            output.stddev
        )

means = torch.cat(means).numpy()
print(f"Means: {means}")
stds = torch.cat(stds).numpy()
print(f"Uncertainties: {stds}")

In [ ]:
predictions = {}
for index, title in enumerate(real_df['title']):
    predictions[title] = [float(means[index]), float(stds[index])]

print(predictions)

In [ ]:
sorted_predictions = dict(sorted(predictions.items(), key=lambda item: item[1][0], reverse=True))
pprint(sorted_predictions, indent=4, sort_dicts=False)

In [ ]:
target_dir = "..\data\processed"
file_name = "dropped_predictions.json" # change depending on what your model is training on
file_path = os.path.join(target_dir, file_name)

with open(file_path, "w", encoding="utf-8") as file:
    json.dump(sorted_predictions, file, indent=4)